# ¿Cuánta incertidumbre tienen las correlaciones?
El intervalo del 95 % es marginal. La prueba de máximo absoluto ajusta los 15 contrastes Pearson bajo una hipótesis de fase relativa aleatoria. La rotación y los bloques suponen estabilidad aproximada. Spearman queda descriptivo.
Ejecuta las celdas en orden. Las tablas CSV y figuras se guardan en la carpeta de trabajo.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/gretadive/correlacion_actividad_solar_elnino/751cd7c187616d8b7ed7b175280dc80a4e999a29/data/processed/dataset_integrado_precipitacion.csv"
archivo = Path("datos.csv")
df = pd.read_csv(archivo if archivo.exists() else url, parse_dates=["fecha"])
df = df.set_index("fecha").sort_index().asfreq("MS")
variables = ["ICEN", "ONI", "precipitacion_mm"]
assert len(df) == 917 and df[variables + ["SSN"]].notna().all().all()
desfases = [0, 3, 6, 12, 24]


In [ ]:
from scipy.stats import norm
# Diagnóstico por mitades y persistencia mensual de las series.
diag = []
for nombre in ["SSN"] + variables:
    for etapa, serie in zip(["1950–1987","1988–2026"],[df.loc[:"1987",nombre],df.loc["1988":,nombre]]):
        diag.append([nombre,etapa,len(serie),serie.mean(),serie.std(),serie.autocorr(1),serie.autocorr(12)])
diagnostico = pd.DataFrame(diag, columns=["variable","periodo","n","media","DE","ACF1","ACF12"])
diagnostico.to_csv("12_diagnostico.csv",index=False)
print(diagnostico.round(2).to_string(index=False))
# Bootstrap circular de bloques de pares ya alineados. IC marginal para Pearson.
# Prueba familiar por rotaciones de SSN: conserva su orden y usa el máximo |r| de 15 contrastes.
def coeficientes(solar):
    return np.array([np.corrcoef(solar[:len(df)-k],df[v].to_numpy()[k:])[0,1]
                     for v in variables for k in desfases])
x = df.SSN.to_numpy()
observado = coeficientes(x)
maximos = np.array([np.abs(coeficientes(np.roll(x,i))).max() for i in range(len(x))])
rng = np.random.default_rng(2026)
filas = []
for j,(v,k) in enumerate((v,k) for v in variables for k in desfases):
    y = df[v].to_numpy()[k:]
    n = len(y)
    for bloque in [12,24,48]:
        muestras = []
        for lote in range(20):
            inicio = rng.integers(n,size=(100,int(np.ceil(n/bloque)),1))
            idx = ((inicio + np.arange(bloque)) % n).reshape(100,-1)[:,:n]
            a,b = x[:n][idx],y[idx]
            a -= a.mean(axis=1,keepdims=True)
            b -= b.mean(axis=1,keepdims=True)
            muestras.extend((a*b).sum(1)/np.sqrt((a*a).sum(1)*(b*b).sum(1)))
        li,ls = np.quantile(muestras,[.025,.975])
        filas.append([v,k,bloque,observado[j],li,ls,np.mean(maximos >= abs(observado[j])-1e-12)])
ic = pd.DataFrame(filas, columns=["variable","meses","bloque","Pearson","IC_inf","IC_sup","p_max_rotacion"])
ic.to_csv("12_incertidumbre.csv", index=False)
print(ic.query("bloque==24").round(3).to_string(index=False))


        variable   periodo   n  media    DE  ACF1  ACF12
             SSN 1950–1987 456 101.03 79.78  0.94   0.75
             SSN 1988–2026 461  81.72 65.67  0.93   0.76
            ICEN 1950–1987 456   0.03  1.01  0.96  -0.15
            ICEN 1988–2026 461  -0.02  1.03  0.96  -0.02
             ONI 1950–1987 456   0.03  0.75  0.97  -0.11
             ONI 1988–2026 461  -0.00  0.82  0.97  -0.02
precipitacion_mm 1950–1987 456 137.24 74.76  0.70   0.71
precipitacion_mm 1988–2026 461 144.23 82.89  0.70   0.67
        variable  meses  bloque  Pearson  IC_inf  IC_sup  p_max_rotacion
            ICEN      0      24    0.092  -0.052   0.236           0.952
            ICEN      3      24    0.125  -0.027   0.274           0.718
            ICEN      6      24    0.143  -0.017   0.303           0.562
            ICEN     12      24    0.123  -0.055   0.290           0.736
            ICEN     24      24    0.120  -0.057   0.304           0.759
             ONI      0      24    0.081  -0.099 